In [39]:
import os

In [40]:
%pwd

'c:\\Users\\DELL\\Documents\\Complete-ml-project\\Complete-ml-deployment'

In [41]:
os.chdir("c:/Users/DELL/Documents/Complete-ml-project/Complete-ml-deployment")

In [42]:
%pwd

'c:\\Users\\DELL\\Documents\\Complete-ml-project\\Complete-ml-deployment'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str

In [44]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml, create_directories

In [45]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluator_config(self) -> ModelEvaluatorConfig:
        config = self.config.model_evaluation
        params = self.params.Elasticnet
        schema = self.schema.TargetColumn

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluatorConfig(
            root_dir=Path(config.root_dir),
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_path=config.metric_file_path,
            target_column=schema.name,
        )

        return model_evaluation_config

In [46]:
import os
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from urllib.parse import urlparse
import numpy as np
import joblib

In [47]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluatorConfig):
        self.config = config

    def evaluate_model(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2_square = r2_score(actual, pred)
        return rmse, mae, r2_square

    def save_results(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop(columns=[self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]

        predicted_qualities = model.predict(test_x)

        (rmse, mae, r2_square) = self.evaluate_model(test_y, predicted_qualities)

        #savingmetrics as local
        scores = {
            "RMSE": rmse,
            "MAE": mae,
            "R2_Square": r2_square
        }

        save_json(path=path(self.config.metric_file_name), data=scores)


In [48]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluator_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.save_results()
except Exception as e:
    raise e

[2025-10-22 00:25:38,360: INFO: common]: yaml file: config\config.yaml loaded successfully
[2025-10-22 00:25:38,365: INFO: common]: yaml file: params.yaml loaded successfully
[2025-10-22 00:25:38,374: INFO: common]: yaml file: schema.yaml loaded successfully
[2025-10-22 00:25:38,379: INFO: common]: created directory at: artifacts


[2025-10-22 00:25:38,383: INFO: common]: created directory at: artifacts/model_evaluation


BoxKeyError: "'ConfigBox' object has no attribute 'metric_file_path'"